# Episode 11: Tokens and Embeddings
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2011%20-%20Tokens%20and%20Embeddings.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 11, one step per section. It needs `numpy` and `tiktoken`; the last section also downloads GPT-2 small's published weights (548 MB, once). Choose **Runtime, Run all**. If you only want the toy parts, stop before the section titled "Real scale".

Covers: the real GPT-2 tokenization of the running sentence, a toy byte-pair-encoding merge loop with every count, encoding new words, the embedding lookup as a one-hot matrix product (Episode 3's product rule), how special tokens change N, why the same total gives different losses under different tokenizers, and the real GPT-2 embedding table (size, nearest rows, the average-cosine baseline). The toy embedding numbers and the 20-nat total are illustrative.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

In [1]:
import importlib, subprocess, sys
for pkg in ["tiktoken", "safetensors", "huggingface_hub"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import tiktoken
from math import exp
print("numpy", np.__version__, "| tiktoken", tiktoken.__version__)

numpy 2.5.3 | tiktoken 0.14.0


## The running sentence under a real tokenizer
GPT-2's byte-pair tokenizer. The tokenizer adds no start or end-of-text token.

In [2]:
SENT = "The cat sat on the mat because it was tired."
enc = tiktoken.get_encoding("gpt2")
ids = enc.encode(SENT)
pieces = [enc.decode([i]) for i in ids]
print("ids   ", ids)
print("pieces", pieces)
n_real = len(ids)
print("n =", n_real, "tokens;", len(SENT.split()), "words;", len(SENT.encode()), "UTF-8 bytes")
assert ids == [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13] and n_real == 11
assert 50256 not in ids                                 # no end-of-text token added
assert len(SENT.split()) == 10 and len(SENT.encode()) == 44
assert enc.encode(" because") == [780]                  # "because" is one token (with its leading space)

for name in ["gpt2", "p50k_base", "cl100k_base", "o200k_base"]:
    e = tiktoken.get_encoding(name)
    print(f"{name:12s} n = {len(e.encode(SENT))}")
    assert len(e.encode(SENT)) == 11

# the typo, and "tired" typed alone (no space before it)
print(" becuase ->", [enc.decode([i]) for i in enc.encode(" becuase")])
print("tired    ->", [enc.decode([i]) for i in enc.encode("tired")])
assert len(enc.encode(" becuase")) == 3 and len(enc.encode("tired")) == 2

# vocabulary: 256 bytes + 50,000 merges + 1 special token
assert enc.n_vocab == 50257 == 256 + 50_000 + 1 and enc.eot_token == 50256

# ' because' is merge number 525, built from ' bec' and 'ause'
ranks = enc._mergeable_ranks
def last_split(tok):
    r = ranks[tok]
    parts = [bytes([b]) for b in tok]
    while len(parts) > 2:
        best = None
        for i in range(len(parts) - 1):
            pr = ranks.get(parts[i] + parts[i + 1])
            if pr is not None and pr < r and (best is None or pr < best[0]):
                best = (pr, i)
        if best is None:
            break
        parts[best[1]:best[1] + 2] = [parts[best[1]] + parts[best[1] + 1]]
    return parts
a, b = last_split(b" because")
print("' because' has ID", ranks[b" because"], "= merge number", ranks[b" because"] - 255, "from", a, "+", b)
assert ranks[b" because"] == 780 and 780 - 255 == 525 and (a, b) == (b" bec", b"ause")

ids    [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13]
pieces ['The', ' cat', ' sat', ' on', ' the', ' mat', ' because', ' it', ' was', ' tired', '.']
n = 11 tokens; 10 words; 44 UTF-8 bytes
gpt2         n = 11
p50k_base    n = 11


cl100k_base  n = 11


o200k_base   n = 11
 becuase -> [' bec', 'u', 'ase']
tired    -> ['t', 'ired']
' because' has ID 780 = merge number 525 from b' bec' + b'ause'


## Special tokens and N
Episode 10's rule: N is the stream length minus 1. A special token changes the stream length, not the rule.

In [3]:
rows = []
for label, stream in [("Sentence alone (tokenizer output)", n_real),
                      ("End-of-text appended (nanoGPT style)", n_real + 1),
                      ("Start token in front instead", n_real + 1)]:
    rows.append((label, stream, stream - 1, stream - 1))
    print(f"{label:38s} stream = {stream:2d}   rows of U in a training pass = {stream-1:2d}   N = {stream-1:2d}")
assert [r[1:] for r in rows] == [(11, 10, 10), (12, 11, 11), (12, 11, 11)]
print("read whole (no training pass), U has", n_real, "rows")

# with <|endoftext|> allowed, the special token is ID 50256; by default tiktoken refuses the string
assert enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}) == [50256]
try:
    enc.encode("<|endoftext|>")
    raise SystemExit("expected a ValueError")
except ValueError:
    print("default encode of '<|endoftext|>' refuses the string, as the episode says")

Sentence alone (tokenizer output)      stream = 11   rows of U in a training pass = 10   N = 10
End-of-text appended (nanoGPT style)   stream = 12   rows of U in a training pass = 11   N = 11
Start token in front instead           stream = 12   rows of U in a training pass = 11   N = 11
read whole (no training pass), U has 11 rows
default encode of '<|endoftext|>' refuses the string, as the episode says


## Step 6: the same total, three tokenizers
Fix the total surprisal at 20 nats (illustrative). L = total ÷ N, with N = tokens − 1. Perplexities under different tokenizers are not comparable.

In [4]:
TOTAL = 20.0
out = {}
for label, n_tok in [("GPT-2 BPE", 11), ("word level (Ep 9)", 10), ("UTF-8 bytes", 44)]:
    N = n_tok - 1
    L = TOTAL / N
    out[label] = (N, round(L, 3), round(exp(L), 2))
    print(f"{label:18s} n = {n_tok:2d}  N = {N:2d}  L = {L:.3f}  PP = {exp(L):.2f}")
assert out == {"GPT-2 BPE": (10, 2.0, 7.39), "word level (Ep 9)": (9, 2.222, 9.23), "UTF-8 bytes": (43, 0.465, 1.59)}
print("total surprisal per byte:", round(TOTAL / 44, 3))
assert round(TOTAL / 44, 3) == 0.455

GPT-2 BPE          n = 11  N = 10  L = 2.000  PP = 7.39
word level (Ep 9)  n = 10  N =  9  L = 2.222  PP = 9.23
UTF-8 bytes        n = 44  N = 43  L = 0.465  PP = 1.59
total surprisal per byte: 0.455


## Step 1: the BPE merge loop (toy)
Illustrative corpus; · is Sennrich et al.'s end-of-word symbol (GPT-2 keeps the space at the front of the word instead). Words and counts: cat 5, mat 3, sat 2, mats 1.

In [5]:
corpus = {"c a t ·": 5, "m a t ·": 3, "s a t ·": 2, "m a t s ·": 1}

def get_stats(vocab):
    pairs = {}
    for word, f in vocab.items():
        s = word.split()
        for i in range(len(s) - 1):
            pairs[(s[i], s[i + 1])] = pairs.get((s[i], s[i + 1]), 0) + f
    return pairs

def merge_vocab(pair, vocab):
    out = {}
    for word, f in vocab.items():
        s, o, i = word.split(), [], 0
        while i < len(s):
            if i < len(s) - 1 and (s[i], s[i + 1]) == pair:
                o.append(s[i] + s[i + 1]); i += 2
            else:
                o.append(s[i]); i += 1
        out[" ".join(o)] = f
    return out

symbols = lambda vv: sum(len(w.split()) * f for w, f in vv.items())
v, merges, winners, sizes = dict(corpus), [], [], []
for step in range(1, 6):
    stats = get_stats(v)
    ordered = sorted(stats.items(), key=lambda kv: -kv[1])
    assert ordered[0][1] > ordered[1][1], "tie: a tie needs a stated rule"      # no ties in this toy
    print(f"step {step}: " + ", ".join(f"({a},{b})={c}" for (a, b), c in ordered))
    pair = ordered[0][0]
    merges.append(pair); winners.append(ordered[0][1])
    v = merge_vocab(pair, v)
    sizes.append(symbols(v))
    print("   merge", pair, "-> corpus:", v)

assert symbols(corpus) == 45
assert winners == [11, 10, 5, 3, 2]
assert merges == [("a", "t"), ("at", "·"), ("c", "at·"), ("m", "at·"), ("s", "at·")]
assert sizes[:4] == [34, 24, 19, 16]                      # 45 symbols fall to 16 after four merges
print("symbols after merges 1 to 5:", sizes)

step 1: (a,t)=11, (t,·)=10, (c,a)=5, (m,a)=4, (s,a)=2, (t,s)=1, (s,·)=1
   merge ('a', 't') -> corpus: {'c at ·': 5, 'm at ·': 3, 's at ·': 2, 'm at s ·': 1}
step 2: (at,·)=10, (c,at)=5, (m,at)=4, (s,at)=2, (at,s)=1, (s,·)=1
   merge ('at', '·') -> corpus: {'c at·': 5, 'm at·': 3, 's at·': 2, 'm at s ·': 1}
step 3: (c,at·)=5, (m,at·)=3, (s,at·)=2, (m,at)=1, (at,s)=1, (s,·)=1
   merge ('c', 'at·') -> corpus: {'cat·': 5, 'm at·': 3, 's at·': 2, 'm at s ·': 1}
step 4: (m,at·)=3, (s,at·)=2, (m,at)=1, (at,s)=1, (s,·)=1
   merge ('m', 'at·') -> corpus: {'cat·': 5, 'mat·': 3, 's at·': 2, 'm at s ·': 1}
step 5: (s,at·)=2, (m,at)=1, (at,s)=1, (s,·)=1
   merge ('s', 'at·') -> corpus: {'cat·': 5, 'mat·': 3, 'sat·': 2, 'm at s ·': 1}
symbols after merges 1 to 5: [34, 24, 19, 16, 14]


In [6]:
toy_vocab = ["c", "a", "t", "m", "s", "·", "at", "at·", "cat·", "mat·"]       # |V| = 10, IDs 0 to 9
assert len(toy_vocab) == 10

def encode_word(word, ms):
    s = list(word) + ["·"]
    for a_, b_ in ms:                                     # replay the merges in the order they were learned
        i = 0
        while i < len(s) - 1:
            if (s[i], s[i + 1]) == (a_, b_):
                s[i:i + 2] = [a_ + b_]
            else:
                i += 1
    return s

first4 = merges[:4]
for w in ["stat", "cat", "mats", "sat"]:
    print(f"{w:5s} ->", encode_word(w, first4))
assert encode_word("stat", first4) == ["s", "t", "at·"]
assert encode_word("cat", first4) == ["cat·"]
assert encode_word("mats", first4) == ["m", "at", "s", "·"]
assert encode_word("sat", first4) == ["s", "at·"]          # check question (a): 2 tokens
assert encode_word("sat", merges) == ["s" + "at·"]         # with the fifth merge: 1 token

stat  -> ['s', 't', 'at·']
cat   -> ['cat·']
mats  -> ['m', 'at', 's', '·']
sat   -> ['s', 'at·']


## Step 3: the lookup as a one-hot matrix product
Episode 3's product rule (m×k)(k×r), with m = 3 tokens, k = |V| = 10, r = d = 2.

In [7]:
rng = np.random.default_rng(10)
W_E = np.round(rng.normal(size=(10, 2)), 1)               # illustrative, d = 2; not Episode 2's vectors
for i, t in enumerate(toy_vocab):
    print(f"{i} {t:5s} {tuple(float(x) for x in W_E[i])}")

toks = encode_word("stat", first4)
tid = [toy_vocab.index(t) for t in toks]
O = np.zeros((len(tid), 10), dtype=int)
for r, j in enumerate(tid):
    O[r, j] = 1
print("IDs", tid)
for r in range(O.shape[0]):
    print("O row", r + 1, "".join(map(str, O[r])))
U = O @ W_E
print("U = O W_E =", U.tolist(), "shape", U.shape)
assert tid == [4, 2, 7]
assert O.shape == (3, 10) and W_E.shape == (10, 2) and U.shape == (3, 2)
assert np.array_equal(U, W_E[tid])                         # each row of U is a copy of one row of W_E
assert U.tolist() == [[0.5, -0.5], [-0.2, 0.1], [-1.0, -1.1]]
assert (O.sum(axis=1) == 1).all()                          # one nonzero term per sum: U_ij = sum_k O_ik (W_E)_kj

0 c     (-1.1, -0.7)
1 a     (-0.8, 0.3)
2 t     (-0.2, 0.1)
3 m     (0.8, 0.9)
4 s     (0.5, -0.5)
5 ·     (-0.8, -0.8)
6 at    (-0.3, -0.1)
7 at·   (-1.0, -1.1)
8 cat·  (0.3, -1.9)
9 mat·  (-0.2, 0.4)
IDs [4, 2, 7]
O row 1 0000100000
O row 2 0010000000
O row 3 0000000100
U = O W_E = [[0.5, -0.5], [-0.2, 0.1], [-1.0, -1.1]] shape (3, 2)


## Real scale: GPT-2 small's embedding table
This section downloads GPT-2 small's published weights (548 MB, once) and reads only the token-embedding table `wte.weight`. Skip it if you are on a slow connection.

In [8]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
path = hf_hub_download("openai-community/gpt2", "model.safetensors", local_dir="gpt2_weights")
f = safe_open(path, framework="np")
W = f.get_tensor("wte.weight")
V_, d_ = W.shape
print("wte shape", W.shape, W.dtype)
assert (V_, d_) == (50257, 768)

# the lookup as a product, and as plain row indexing
O_big = np.zeros((n_real, V_), dtype=np.float32)
O_big[np.arange(n_real), ids] = 1.0
U_big = O_big @ W
assert O_big.shape == (11, 50257) and U_big.shape == (11, 768) and np.allclose(U_big, W[ids], atol=1e-6)
macs, copies = n_real * V_ * d_, n_real * d_
print("product costs", macs, "multiply-adds; copying rows costs", copies, "; ratio", macs // copies)
assert macs == 424_571_136 and copies == 8_448 and macs // copies == V_

# size of the table and its share of the parameters (the attention-mask buffers h.N.attn.bias are not parameters)
n_table = V_ * d_
total_params = sum(int(np.prod(f.get_slice(k).get_shape())) for k in f.keys() if not k.endswith(".attn.bias"))
print("table entries", n_table, "; parameters", total_params, "; share", round(100 * n_table / total_params, 1), "%")
assert n_table == 38_597_376 and total_params == 124_439_808 and round(100 * n_table / total_params, 1) == 31.0

wte shape (50257, 768) float32
product costs 424571136 multiply-adds; copying rows costs 8448 ; ratio 50257
table entries 38597376 ; parameters 124439808 ; share 31.0 %


In [9]:
Wd = W.astype(np.float64)
unit = Wd / np.linalg.norm(Wd, axis=1, keepdims=True)

def tid1(s):
    t = enc.encode(s); assert len(t) == 1, s
    return t[0]

def nearest(s, k=5):
    j0 = tid1(s)
    sims = unit @ unit[j0]
    order = [j for j in np.argsort(-sims) if j != j0][:k]
    return [(enc.decode([j]), round(float(sims[j]), 3)) for j in order]

for w in [" cat", " tired", " mat"]:
    print("nearest to", repr(w), nearest(w))
cos = lambda a_, b_: float(unit[tid1(a_)] @ unit[tid1(b_)])
for a_, b_ in [(" cat", " dog"), (" cat", " mat"), (" cat", " tired"), (" cat", " because")]:
    print(f"cos({a_!r}, {b_!r}) = {cos(a_, b_):.3f}")
assert nearest(" cat")[-1][0] == " dog" and round(cos(" cat", " dog"), 3) == 0.550
assert nearest(" cat")[0] == (" Cat", 0.707) and nearest(" tired")[0][0] == " weary"
assert round(cos(" cat", " mat"), 3) == 0.264 and round(cos(" cat", " tired"), 3) == 0.222

# average cosine over all pairs of distinct rows (exact): (K * |mean of unit rows|^2 - 1) / (K - 1)
K = unit.shape[0]
mu = unit.mean(0)
avg_all = (K * float(mu @ mu) - 1) / (K - 1)
Wc = Wd - Wd.mean(0)
unit_c = Wc / np.linalg.norm(Wc, axis=1, keepdims=True)
mu_c = unit_c.mean(0)
avg_c = (K * float(mu_c @ mu_c) - 1) / (K - 1)
print("average cosine over all pairs:", round(avg_all, 3), "| after subtracting the mean row:", round(avg_c, 3))
assert round(avg_all, 3) == 0.268 and abs(avg_c) < 5e-4

# how much of that average the 3 largest dimensions carry
contrib = ((K * mu) ** 2 - (unit ** 2).sum(0)) / (K * (K - 1))
assert abs(contrib.sum() - avg_all) < 1e-9
top = np.argsort(-contrib)
share3 = float(contrib[top[:3]].sum() / avg_all)
print("3 largest dimensions carry", round(100 * share3, 1), "% of the average; equal shares would give", round(100 * 3 / 768, 1), "%")
assert round(100 * share3, 1) == 8.5 and round(100 * 3 / 768, 1) == 0.4

nearest to ' cat' [(' Cat', 0.707), (' cats', 0.693), ('Cat', 0.642), ('cat', 0.603), (' dog', 0.55)]
nearest to ' tired' [(' weary', 0.665), (' bored', 0.642), (' exhausted', 0.606), (' frustrated', 0.578), (' fatig', 0.567)]
nearest to ' mat' [(' Mat', 0.632), (' mats', 0.584), ('Mat', 0.567), ('mat', 0.512), (' matrix', 0.472)]
cos(' cat', ' dog') = 0.550
cos(' cat', ' mat') = 0.264
cos(' cat', ' tired') = 0.222
cos(' cat', ' because') = 0.234


average cosine over all pairs: 0.268 | after subtracting the mean row: 0.0
3 largest dimensions carry 8.5 % of the average; equal shares would give 0.4 %


## Check question
(a) With the four toy merges, "sat" is 2 tokens (checked above; a fifth merge would make it 1). (b) A 7-token document gets an end-of-text token appended: stream 8, so U has 7 rows and N = 7. (c) With |V| = 50,257 and d = 768: O is 7 × 50,257, W_E is 50,257 × 768, U is 7 × 768, and the table holds 38,597,376 numbers.

In [10]:
stream = 7 + 1
rows_U, N_b = stream - 1, stream - 1
assert (rows_U, N_b) == (7, 7)
shapes = {"O": (rows_U, V_), "W_E": (V_, d_), "U": (rows_U, d_)}
print(shapes, "table entries", V_ * d_)
assert shapes == {"O": (7, 50257), "W_E": (50257, 768), "U": (7, 768)} and V_ * d_ == 38_597_376
print("All Episode 11 checks passed.")

{'O': (7, 50257), 'W_E': (50257, 768), 'U': (7, 768)} table entries 38597376
All Episode 11 checks passed.
